In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('dirty_cafe_sales.csv')

In [3]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


###  Convert columns to its correct type like  Transaction Date  Quantity, Price Per Unit and Total Spent

In [5]:
data['Transaction Date'] = pd.to_datetime(data['Transaction Date'], errors='coerce')

num_cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in num_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9667 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    7421 non-null   str           
 6   Location          6735 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


# tak the columns and start to processing (Quantity)


In [7]:
data['Quantity'].unique()

array([ 2.,  4.,  5.,  3.,  1., nan])

# the mwdin_data = Total Spent / Price Per Unit, we can use this to fill the missing values in the quantity column

# if price per unit is empty or nan, we can not calculate the quantity, so we will fill quantity with the median of the quantity column

In [8]:
calculated_qty = np.where(
    (data['Price Per Unit'].notna()) & (data['Price Per Unit'] != 0) & (data['Total Spent'].notna()),
    data['Total Spent'] / data['Price Per Unit'],
    np.nan
)

calculated_qty = pd.Series(calculated_qty, index=data.index)

median_quantity = data['Quantity'].median()

data['Quantity'] = data['Quantity'].fillna(calculated_qty).fillna(median_quantity)  


In [9]:
data['Quantity'].isna().sum()

np.int64(0)

In [10]:
data['Quantity'].value_counts()

Quantity
5.0    2106
2.0    2052
3.0    1984
4.0    1937
1.0    1921
Name: count, dtype: int64

In [11]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


# take the columns and start to processing (Price Per Unit)


In [12]:
data['Price Per Unit'].isna().sum()

np.int64(533)

In [13]:
data['Price Per Unit'].unique()

array([2. , 3. , 1. , 5. , 4. , 1.5, nan])

# price per unit = [Total_Spent] / [Quantity] and fill missing values and if the quantity is missing or total spent is missing, full the price per unit using the median price per unit and Avoid divsion yb zero 

In [14]:

price_per_unit_calc = data['Total Spent'] / data['Quantity']


price_per_unit_calc = price_per_unit_calc.replace([float('inf'), -float('inf')], float('nan'))

median_price_per_unit = price_per_unit_calc.median()

if 'Price Per Unit' not in data.columns:
    data['Price Per Unit'] = price_per_unit_calc

data['Price Per Unit'] = data['Price Per Unit'].fillna(price_per_unit_calc).fillna(median_price_per_unit)

In [15]:
data['Price Per Unit'].isna().sum()

np.int64(0)

In [16]:
data['Price Per Unit'].unique()

array([2.        , 3.        , 1.        , 5.        , 4.        ,
       1.5       , 6.66666667, 5.33333333, 1.66666667, 0.33333333,
       8.33333333, 2.66666667, 0.66666667, 1.33333333, 2.5       ])

In [17]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


# take the columns and start to processing (Total Spent)


In [18]:
data['Total Spent'].isna().sum()

np.int64(502)

In [19]:
data['Total Spent'].unique()

array([ 4. , 12. ,  nan, 10. , 20. ,  9. , 16. , 15. , 25. ,  8. ,  5. ,
        3. ,  6. ,  2. ,  1. ,  7.5,  4.5,  1.5])

In [20]:
Total = data['Price Per Unit'] * data['Quantity']

In [21]:
data['Total Spent'] = data['Total Spent'].fillna(Total)

In [22]:
data['Total Spent'].isna().sum()

np.int64(0)

In [23]:
data['Total Spent'].unique()

array([ 4. , 12. , 10. , 20. ,  9. , 16. , 15. , 25. ,  8. ,  5. ,  3. ,
        6. ,  2. ,  1. ,  7.5,  4.5,  1.5])

In [24]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


# take the columns and start to processing (Payment Method)


In [25]:
data['Payment Method'].isna().sum()

np.int64(2579)

In [26]:
data['Payment Method'].unique()

<StringArray>
['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR', nan]
Length: 6, dtype: str

In [27]:
mode_payment_method = data['Payment Method'].mode()[0] 
data['Payment Method'] = data['Payment Method'].fillna(mode_payment_method)

In [28]:
data['Payment Method'].unique()

<StringArray>
['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR']
Length: 5, dtype: str

In [29]:
data['Payment Method'] = data['Payment Method'].replace('UNKNOWN', mode_payment_method)
data['Payment Method'] = data['Payment Method'].replace('ERROR', mode_payment_method)

In [30]:
data['Payment Method'].unique()

<StringArray>
['Credit Card', 'Cash', 'Digital Wallet']
Length: 3, dtype: str

In [31]:
data['Payment Method'].isna().sum()

np.int64(0)

In [32]:
data['Payment Method'].value_counts()

Payment Method
Digital Wallet    5469
Credit Card       2273
Cash              2258
Name: count, dtype: int64

In [33]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


# tak the columns and start to processing (Location)


In [34]:
data['Location'].isna().sum()

np.int64(3265)

In [35]:
data['Location'].unique()

<StringArray>
['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR']
Length: 5, dtype: str

In [36]:
data['Location'].value_counts()

Location
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

In [37]:
mode_lication = data['Location'].mode()[0]

In [38]:
data['Location'] = data['Location'].fillna(mode_lication)

In [39]:
data['Location'].value_counts()

Location
Takeaway    6287
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

In [40]:
data['Location'] = data['Location'].replace('UNKNOWN', mode_lication)
data['Location'] = data['Location'].replace('ERROR', mode_lication)

In [41]:
data['Location'].value_counts()

Location
Takeaway    6983
In-store    3017
Name: count, dtype: int64

In [42]:
data.isna().sum()

Transaction ID        0
Item                333
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

In [43]:
data['Item'].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

# any item is null or empty or UNKNOWN or ERROR, we will see to the price per unit and fill the item with the most common item with that price per unit, if there is no such item, we will fill it with the most common item in the dataset

In [44]:

valid_items = data[data['Item'].notna() & ~data['Item'].isin(['UNKNOWN', 'ERROR', ''])]
mode_item = valid_items['Item'].mode()[0] if not valid_items.empty else 'Unknown'

data['Price_Rounded'] = data['Price Per Unit'].round(2)
price_to_mode = data.groupby('Price_Rounded')['Item'].agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan)

def fill_item(row):
    if pd.isna(row['Item']) or row['Item'] in ['UNKNOWN', 'ERROR', '']:
        price = row['Price Per Unit']
        if pd.notna(price):
            mode_for_price = price_to_mode.get(round(price, 2))
            if pd.notna(mode_for_price):
                return mode_for_price
        return mode_item
    return row['Item']

data['Item'] = data.apply(fill_item, axis=1)
data.drop('Price_Rounded', axis=1, inplace=True)


In [45]:
data['Item'].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',    'Juice',
 'Sandwich',      'Tea',  'UNKNOWN']
Length: 9, dtype: str

In [46]:
data['Item'].isna().sum()

np.int64(0)

In [47]:
data['Item'].value_counts()

Item
Juice       1422
Sandwich    1359
Coffee      1291
Salad       1272
Cookie      1213
Tea         1207
Cake        1139
Smoothie    1096
UNKNOWN        1
Name: count, dtype: int64

In [48]:
mode_item = data['Item'].mode()[0]
data['Item'] = data['Item'].replace('UNKNOWN', mode_item)

In [49]:
data['Item'].value_counts()

Item
Juice       1423
Sandwich    1359
Coffee      1291
Salad       1272
Cookie      1213
Tea         1207
Cake        1139
Smoothie    1096
Name: count, dtype: int64

In [50]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [51]:
data.isna().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

# processing the date

In [52]:
data['Transaction Date'].unique()

<DatetimeArray>
['2023-09-08 00:00:00', '2023-05-16 00:00:00', '2023-07-19 00:00:00',
 '2023-04-27 00:00:00', '2023-06-11 00:00:00', '2023-03-31 00:00:00',
 '2023-10-06 00:00:00', '2023-10-28 00:00:00', '2023-07-28 00:00:00',
 '2023-12-31 00:00:00',
 ...
 '2023-08-01 00:00:00', '2023-01-20 00:00:00', '2023-11-11 00:00:00',
 '2023-02-13 00:00:00', '2023-07-30 00:00:00', '2023-02-17 00:00:00',
 '2023-05-20 00:00:00', '2023-11-05 00:00:00', '2023-03-27 00:00:00',
 '2023-07-03 00:00:00']
Length: 366, dtype: datetime64[us]

In [53]:
data['Transaction Date'].value_counts()

Transaction Date
2023-02-06    40
2023-06-16    40
2023-03-13    39
2023-07-21    39
2023-07-24    39
              ..
2023-09-24    15
2023-07-30    15
2023-03-11    14
2023-07-22    14
2023-02-17    14
Name: count, Length: 365, dtype: int64

In [54]:
data['Transaction Date'].isna().sum()

np.int64(460)

In [55]:
# drop the null valuse from Transaction Date

data = data.dropna(subset=['Transaction Date'])

In [56]:
data['Transaction Date'].isna().sum()

np.int64(0)

In [57]:
data.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

# make function to normalizing the date and take the monthe after that distribution it into four classes according to the condition, and then ceate a column in the data file that stores the values.

In [58]:
def get_season(date):
    if pd.isna(date):
        return 'Unknown'
    month = date.month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:  # 9, 10, 11
        return 'Fall'


data['season'] = data['Transaction Date'].apply(get_season)

In [59]:
data.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
season              0
dtype: int64

In [60]:
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,season
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,Fall
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,Spring
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,Summer
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27,Spring
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,Summer


# safe data that cleaned with string sum values in sum columne 
# with name data_cleand1

In [61]:
data_cleaned = data.reset_index()

In [62]:
data_cleaned.to_csv("data_Cleand1.csv" , index=False)

# safe data after changed the string values to number


In [63]:
data['Location'].unique()

<StringArray>
['Takeaway', 'In-store']
Length: 2, dtype: str

# Encode the values from string value to number

In [64]:
data['Location'] = data['Location'].replace({'Takeaway':0, 'In-store':1})
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,season
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,0,2023-09-08,Fall
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,1,2023-05-16,Spring
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,1,2023-07-19,Summer
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,0,2023-04-27,Spring
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,1,2023-06-11,Summer


In [65]:
data['Payment Method'].unique()

<StringArray>
['Credit Card', 'Cash', 'Digital Wallet']
Length: 3, dtype: str

# Note:
#  I did not encode the values ​​in the rest of the columns because the values ​​are more than two words and sometimes it affects the display or mathematical operations.

# safe data to csv fill

In [67]:
# safe data to csv fill after filltering 

data.to_csv('data_Cleaned.csv', index=False)